# Module 3.2 — Multivariate Time Series Analysis
## When Assets Speak to Each Other

---

### On the Interconnectedness of Markets

No asset trades in isolation. The price of crude oil shapes the fate of airline stocks. The yield on a 10-year Treasury ripples through equity valuations, mortgage rates, and currency pairs. A semiconductor shortage in Taiwan surfaces months later in the earnings of automobile manufacturers in Detroit. Markets are not a collection of independent processes—they are a **vast, evolving network of causal and correlational relationships**, and the quant who treats each asset as a univariate time series is willfully ignoring the richest source of signal available.

Multivariate time series analysis is the discipline of modeling these relationships. Where Module 3.1 asked, "Does this series have memory?"—a question about a single instrument's relationship with its own past—this module asks a far more powerful question: **"Does this series have memory of *other* series?"** If the answer is yes, we can exploit that memory for prediction, hedging, and arbitrage.

The tools we develop here form the backbone of some of the most profitable and enduring strategies in quantitative finance. **Pairs trading**—buying an undervalued stock while shorting its overvalued cousin—rests on cointegration theory. **Dynamic hedging**—continuously adjusting hedge ratios as relationships evolve—relies on the Kalman filter. **Cross-asset momentum** and **lead-lag effects** are detected through Granger causality tests. **Risk factor models** that decompose portfolio returns into systematic and idiosyncratic components are, at their core, multivariate time series models.

But power comes with danger. In a world of $N$ assets, there are $N(N-1)/2$ pairwise relationships and an exponentially larger number of multivariate interactions. Most of them are spurious. The **curse of dimensionality** looms over every multivariate model: more parameters to estimate, more opportunities for overfitting, more chances to discover false structure. The discipline required is enormous—you must not only find relationships but prove they are real, stable, and economically meaningful. This module teaches you both the tools and the discipline.

---

### Learning Objectives

By the end of this module, you will:

1. **Build and interpret** Vector Autoregression (VAR) models for multi-asset systems
2. **Test for Granger causality** and understand its promise and limitations
3. **Detect cointegration** using the Engle-Granger and Johansen tests
4. **Construct** error correction models for mean-reverting spreads
5. **Implement** a Kalman filter for dynamic hedge ratio estimation
6. **Design** a pairs trading strategy grounded in cointegration theory
7. **Evaluate** multi-asset forecasting systems with proper out-of-sample discipline

---

### Module Contents

1. **Vector Autoregression (VAR)** — Modeling systems of interacting time series
2. **Granger Causality** — Does knowing $X$ help predict $Y$?
3. **Cointegration** — When non-stationary series share a common stochastic trend
4. **Error Correction Models** — Exploiting the pull of equilibrium
5. **The Kalman Filter** — Optimal estimation in a world of noise
6. **Pairs Trading** — The canonical cointegration strategy
7. **Multi-Asset Forecasting** — Putting the system to work

---

*"The test of a first-rate intelligence is the ability to hold two opposed ideas in mind at the same time and still retain the ability to function." — F. Scott Fitzgerald*

*The test of a first-rate quant is the ability to model the interactions of dozens of assets simultaneously and still retain the ability to trade.*

In [ ]:
# ========================================
# Initial Setup and Configuration
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings

from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

PLOT_CONFIG = {
    'figure.figsize': (14, 7),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 100,
    'lines.linewidth': 2,
}
plt.rcParams.update(PLOT_CONFIG)

COLORS = {
    'asset_a': '#1565C0',
    'asset_b': '#C62828',
    'spread': '#2E7D32',
    'forecast': '#F57C00',
    'kalman': '#6A1B9A',
    'signal': '#00838F',
    'pnl': '#2E7D32',
    'neutral': '#455A64',
    'confidence': '#90CAF9',
}

print("=" * 80)
print(" " * 12 + "MODULE 3.2: MULTIVARIATE TIME SERIES ANALYSIS")
print("=" * 80)
print(f"✓ Environment configured successfully")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n📚 Ready to explore the interconnected world of markets!\n")

---

## 1. Data Generation: A Multi-Asset Universe

### Building a Realistic Laboratory

Before we can study multivariate relationships, we need a universe of assets that exhibit the kinds of dependencies we find in real markets. We construct a system of five simulated assets with carefully designed properties:

- **Stock A and Stock B**: A cointegrated pair (think Coca-Cola and Pepsi, or Exxon and Chevron). Their prices wander independently in the short run but are bound together by a long-run equilibrium. When the spread between them widens, it eventually reverts.
- **Market Index**: A broad index that drives systematic risk across all assets.
- **Sector ETF**: Correlated with the index but with its own idiosyncratic component.
- **Bond Proxy**: A defensive asset with low correlation to equities and occasional negative correlation during risk-off episodes.

This construction ensures we have examples of cointegration, Granger causality, and varying correlation structures—the raw materials for every technique in this module.

In [ ]:
# ========================================
# Multi-Asset Data Generation
# ========================================

def generate_multivariate_system(n_days: int = 756, seed: int = 42) -> pd.DataFrame:
    """Generate a system of correlated and cointegrated asset prices.

    The system includes:
    - A cointegrated pair (Stock_A, Stock_B) sharing a common stochastic trend
    - A market index that Granger-causes individual stocks
    - A sector ETF correlated with the index
    - A bond proxy with low/negative equity correlation
    """
    rng = np.random.RandomState(seed)
    dt = 1 / 252

    # Common stochastic trend (shared by cointegrated pair)
    common_trend = np.cumsum(rng.normal(0.0002, 0.008, n_days))

    # Market factor (drives all equities)
    market_shocks = rng.normal(0.0003, 0.01, n_days)
    market_returns = market_shocks

    # Stock A: common trend + idiosyncratic + market beta
    idio_a = np.cumsum(rng.normal(0, 0.005, n_days))
    stock_a_log = np.log(100) + common_trend + idio_a + 0.8 * np.cumsum(market_returns)

    # Stock B: common trend + different idiosyncratic + market beta
    # The cointegrating relationship: Stock_B ≈ 0.6 * Stock_A + constant + noise
    idio_b = np.cumsum(rng.normal(0, 0.004, n_days))
    stock_b_log = np.log(60) + 0.6 * (common_trend - common_trend[0]) + idio_b + 0.7 * np.cumsum(market_returns)

    # Mean-reverting spread noise for cointegration
    spread_noise = np.zeros(n_days)
    phi_spread = 0.95  # AR(1) coefficient < 1 ensures mean-reversion
    for t in range(1, n_days):
        spread_noise[t] = phi_spread * spread_noise[t-1] + rng.normal(0, 0.003)
    stock_b_log += spread_noise

    # Market index
    market_log = np.log(4500) + np.cumsum(market_returns)

    # Sector ETF: correlated with market, with lag
    sector_returns = 0.6 * market_returns + 0.4 * np.roll(market_returns, 1) + rng.normal(0, 0.006, n_days)
    sector_returns[0] = market_returns[0]  # fix roll artifact
    sector_log = np.log(200) + np.cumsum(sector_returns)

    # Bond proxy: low/negative correlation with equities
    bond_returns = -0.15 * market_returns + rng.normal(0.0001, 0.003, n_days)
    bond_log = np.log(100) + np.cumsum(bond_returns)

    dates = pd.bdate_range(start='2022-01-03', periods=n_days)
    df = pd.DataFrame({
        'Stock_A': np.exp(stock_a_log),
        'Stock_B': np.exp(stock_b_log),
        'Market': np.exp(market_log),
        'Sector': np.exp(sector_log),
        'Bond': np.exp(bond_log),
    }, index=dates)
    df.index.name = 'date'

    return df


prices = generate_multivariate_system(n_days=756)
returns = np.log(prices / prices.shift(1)).dropna()

print("=" * 80)
print("MULTI-ASSET UNIVERSE")
print("=" * 80)
print(f"Period: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')} ({len(prices)} days)")
print(f"\n{'Asset':<12} {'Start':>10} {'End':>10} {'Return':>10} {'Ann.Vol':>10}")
print("-" * 56)
for col in prices.columns:
    total_ret = (prices[col].iloc[-1] / prices[col].iloc[0] - 1) * 100
    ann_vol = returns[col].std() * np.sqrt(252) * 100
    print(f"{col:<12} {prices[col].iloc[0]:>10.2f} {prices[col].iloc[-1]:>10.2f} "
          f"{total_ret:>9.1f}% {ann_vol:>9.1f}%")

# Correlation matrix
print("\nReturn Correlation Matrix:")
print(returns.corr().round(3).to_string())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Multi-Asset Universe: The Raw Material",
             fontsize=18, fontweight='bold', y=1.02)

# Normalized prices
ax = axes[0, 0]
normalized = prices / prices.iloc[0] * 100
for col in normalized.columns:
    ax.plot(normalized.index, normalized[col], linewidth=1.5, label=col)
ax.set_title('Normalized Prices (Base = 100)', fontsize=14)
ax.set_ylabel('Index Level')
ax.legend(fontsize=9)

# Cointegrated pair
ax = axes[0, 1]
ax.plot(prices.index, prices['Stock_A'], color=COLORS['asset_a'], label='Stock A')
ax2 = ax.twinx()
ax2.plot(prices.index, prices['Stock_B'], color=COLORS['asset_b'], label='Stock B')
ax.set_title('The Cointegrated Pair: Bound Together', fontsize=14)
ax.set_ylabel('Stock A ($)', color=COLORS['asset_a'])
ax2.set_ylabel('Stock B ($)', color=COLORS['asset_b'])
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

# Rolling correlation
ax = axes[1, 0]
window = 60
rolling_corr_ab = returns['Stock_A'].rolling(window).corr(returns['Stock_B'])
rolling_corr_mb = returns['Market'].rolling(window).corr(returns['Bond'])
ax.plot(rolling_corr_ab.index, rolling_corr_ab, color=COLORS['spread'], label='Stock A vs B')
ax.plot(rolling_corr_mb.index, rolling_corr_mb, color=COLORS['kalman'], label='Market vs Bond')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'{window}-Day Rolling Correlation', fontsize=14)
ax.set_ylabel('Correlation')
ax.legend(fontsize=9)

# Correlation heatmap
ax = axes[1, 1]
mask = np.triu(np.ones_like(returns.corr(), dtype=bool), k=1)
sns.heatmap(returns.corr(), mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Return Correlation Heatmap', fontsize=14)

plt.tight_layout()
plt.show()

---

## 2. Vector Autoregression (VAR)

### 2.1 From Univariate to Multivariate: The VAR Framework

A **Vector Autoregression** model is the multivariate generalization of the AR model. Where an AR($p$) model regresses a single variable on its own lags, a VAR($p$) model regresses a **vector** of variables on the lagged values of **all** variables in the system. For a system of $K$ variables, the VAR($p$) model is:

$$\mathbf{y}_t = \mathbf{c} + \mathbf{A}_1 \mathbf{y}_{t-1} + \mathbf{A}_2 \mathbf{y}_{t-2} + \cdots + \mathbf{A}_p \mathbf{y}_{t-p} + \mathbf{u}_t$$

where $\mathbf{y}_t$ is a $K \times 1$ vector of variables, $\mathbf{A}_i$ are $K \times K$ coefficient matrices, $\mathbf{c}$ is a constant vector, and $\mathbf{u}_t \sim \mathcal{N}(\mathbf{0}, \mathbf{\Sigma})$ is a vector of white noise innovations.

The beauty of VAR is its agnosticism—it does not require you to specify which variables are "dependent" and which are "independent." Every variable is simultaneously a predictor and a predicted. The model discovers the lead-lag structure from the data. But this flexibility has a cost: a VAR($p$) model with $K$ variables estimates $K^2 p + K$ parameters. With 5 assets and 3 lags, that is 80 parameters. Overfitting is a real danger, and the model requires substantial data to estimate reliably.

### 2.2 Granger Causality: Prediction, Not Causation

A variable $X$ is said to **Granger-cause** $Y$ if past values of $X$ contain information useful for predicting $Y$ beyond what past values of $Y$ alone provide. Formally, we test whether the coefficients on lagged $X$ in the VAR equation for $Y$ are jointly zero.

This is a test of **predictive content**, not true causation. If oil prices Granger-cause airline stock returns, it means oil prices help predict airline returns—not that oil *causes* airlines to exist. Nevertheless, Granger causality is powerful for strategy design: if you find that asset $X$ leads asset $Y$ by one or two periods, you can potentially trade $Y$ based on movements in $X$.

### 2.3 Impulse Response Functions

**Impulse Response Functions (IRFs)** trace the effect of a one-standard-deviation shock to one variable on all variables in the system over time. They answer questions like: "If the market index drops 1% today, how do individual stocks, bonds, and sectors respond over the next 20 days?" IRFs are essential for understanding the **dynamic propagation** of shocks through the system and for designing hedging strategies that account for these lagged responses.

In [ ]:
# ========================================
# VAR Modeling and Granger Causality
# ========================================

# --- Fit VAR model ---

# VAR requires stationary data
var_data = returns.copy()

# Select optimal lag order
var_model = VAR(var_data)
lag_selection = var_model.select_order(maxlags=10)

print("=" * 80)
print("VAR LAG ORDER SELECTION")
print("=" * 80)
print(lag_selection.summary().as_text())

optimal_lag = lag_selection.aic
print(f"\n✓ Optimal lag by AIC: {optimal_lag}")

# Fit VAR with optimal lag
var_fit = var_model.fit(maxlags=optimal_lag)

print(f"\n{'=' * 80}")
print(f"VAR({optimal_lag}) MODEL SUMMARY")
print(f"{'=' * 80}")
print(f"Number of equations: {var_fit.neqs}")
print(f"Parameters per equation: {var_fit.k_ar * var_fit.neqs + 1}")
print(f"Total parameters: {var_fit.neqs * (var_fit.k_ar * var_fit.neqs + 1)}")
print(f"Log-likelihood: {var_fit.llf:.2f}")
print(f"AIC: {var_fit.aic:.4f}")
print(f"BIC: {var_fit.bic:.4f}")

# --- Granger Causality Tests ---

print(f"\n{'=' * 80}")
print("GRANGER CAUSALITY MATRIX")
print(f"{'=' * 80}")
print("(p-values: lower = stronger evidence of Granger causality)")
print(f"H0: Row variable does NOT Granger-cause Column variable\n")

gc_matrix = pd.DataFrame(index=returns.columns, columns=returns.columns, dtype=float)

for caused in returns.columns:
    for causing in returns.columns:
        if caused == causing:
            gc_matrix.loc[causing, caused] = np.nan
            continue
        try:
            test_data = returns[[caused, causing]].dropna()
            result = grangercausalitytests(test_data, maxlag=optimal_lag, verbose=False)
            min_pval = min(result[lag][0]['ssr_ftest'][1] for lag in result)
            gc_matrix.loc[causing, caused] = min_pval
        except Exception:
            gc_matrix.loc[causing, caused] = 1.0

print(gc_matrix.round(4).to_string())

print("\nSignificant Granger-causal relationships (p < 0.05):")
for causing in returns.columns:
    for caused in returns.columns:
        if causing != caused:
            pval = gc_matrix.loc[causing, caused]
            if pd.notna(pval) and pval < 0.05:
                print(f"  {causing} → {caused} (p = {pval:.4f})")

# --- Impulse Response Functions ---

irf = var_fit.irf(periods=20)
irf_stderr = irf.stderr()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Impulse Response Functions: How Shocks Propagate",
             fontsize=18, fontweight='bold', y=1.02)

shock_source = 'Market'
responses = ['Stock_A', 'Stock_B', 'Sector', 'Bond']
shock_idx = list(returns.columns).index(shock_source)

for ax, response in zip(axes.flat, responses):
    resp_idx = list(returns.columns).index(response)
    irf_values = irf.irfs[:, resp_idx, shock_idx]
    se = irf_stderr[:, resp_idx, shock_idx]
    lower = irf_values - 1.96 * se
    upper = irf_values + 1.96 * se
    periods = range(len(irf_values))

    ax.plot(periods, irf_values, color=COLORS['asset_a'], linewidth=2)
    ax.fill_between(periods, lower, upper, alpha=0.15, color=COLORS['confidence'])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'Response of {response} to {shock_source} Shock', fontsize=13)
    ax.set_xlabel('Days After Shock')
    ax.set_ylabel('Response')

plt.tight_layout()
plt.show()

# Granger causality heatmap
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
gc_numeric = gc_matrix.astype(float)
sns.heatmap(gc_numeric, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.2, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': 'p-value'})
ax.set_title('Granger Causality p-values\n(Darker = Stronger Causal Signal)', fontsize=14, fontweight='bold')
ax.set_xlabel('Caused (predicted) variable')
ax.set_ylabel('Causing (predictor) variable')
plt.tight_layout()
plt.show()

---

## 3. Cointegration: The Long-Run Bond Between Series

### 3.1 What Is Cointegration?

Cointegration is one of the most profound and practically useful concepts in time series econometrics. Two non-stationary (I(1)) series are **cointegrated** if there exists a linear combination of them that is stationary (I(0)). In plain language: each series can wander freely on its own, but they are **tethered together by a long-run equilibrium**. When the spread between them widens too far, forces (economic, arbitrage, or otherwise) pull them back.

Formally, if $P_t^A \sim I(1)$ and $P_t^B \sim I(1)$, they are cointegrated if there exists a coefficient $\beta$ such that:

$$z_t = P_t^A - \beta P_t^B \sim I(0)$$

The stationary residual $z_t$ is the **cointegrating residual** or **spread**. When $z_t$ deviates from its mean, it is expected to revert—and this mean-reversion is exactly what pairs traders exploit.

**Cointegration is not the same as correlation.** Two assets can be highly correlated but not cointegrated (they move together in the short run but can drift apart permanently). Conversely, two assets can have low short-run correlation but be cointegrated (their spread mean-reverts despite short-run independence). Correlation is a property of returns; cointegration is a property of price levels.

### 3.2 Testing for Cointegration

**Engle-Granger Two-Step Test**:
1. Regress $P_t^A$ on $P_t^B$: estimate $\hat{\beta}$ by OLS
2. Compute residuals: $\hat{z}_t = P_t^A - \hat{\beta} P_t^B$
3. Test $\hat{z}_t$ for stationarity using ADF test

If the residuals are stationary, the series are cointegrated.

**Johansen Test**: A system-level test that can detect multiple cointegrating relationships among $K$ series simultaneously. It estimates the **cointegrating rank**—the number of independent cointegrating vectors. For a system of $K$ I(1) variables, there can be at most $K-1$ cointegrating relationships.

### 3.3 Error Correction Models

Once cointegration is established, the relationship is best modeled with a **Vector Error Correction Model (VECM)**:

$$\Delta \mathbf{y}_t = \boldsymbol{\alpha} \boldsymbol{\beta}' \mathbf{y}_{t-1} + \sum_{i=1}^{p-1} \mathbf{\Gamma}_i \Delta \mathbf{y}_{t-i} + \mathbf{u}_t$$

The term $\boldsymbol{\alpha} \boldsymbol{\beta}' \mathbf{y}_{t-1}$ is the **error correction term**: $\boldsymbol{\beta}' \mathbf{y}_{t-1}$ measures the deviation from equilibrium, and $\boldsymbol{\alpha}$ measures the **speed of adjustment**—how quickly each variable responds to restore equilibrium. Large $|\alpha|$ means fast correction; small $|\alpha|$ means the asset is slow to adjust, which is actionable information for a trader.

In [ ]:
# ========================================
# Cointegration Analysis
# ========================================

print("=" * 80)
print("COINTEGRATION ANALYSIS")
print("=" * 80)

# --- Engle-Granger test for all pairs ---

print("\nEngle-Granger Cointegration Tests (all pairs):")
print(f"{'Pair':<24} {'Test Stat':>10} {'p-value':>10} {'Cointegrated?':>16}")
print("-" * 64)

eg_results = []
for i, col_a in enumerate(prices.columns):
    for col_b in prices.columns[i+1:]:
        score, pval, _ = coint(prices[col_a], prices[col_b])
        is_coint = pval < 0.05
        eg_results.append({
            'pair': f"{col_a} / {col_b}",
            'stat': score,
            'pval': pval,
            'coint': is_coint,
            'col_a': col_a,
            'col_b': col_b,
        })
        marker = "✓ YES" if is_coint else "✗ No"
        print(f"{col_a + ' / ' + col_b:<24} {score:>10.4f} {pval:>10.4f} {marker:>16}")

# --- Johansen test on the full system ---

print(f"\n{'=' * 80}")
print("JOHANSEN COINTEGRATION TEST")
print(f"{'=' * 80}")

johansen_result = coint_johansen(prices, det_order=0, k_ar_diff=2)

print(f"\n{'Rank':<6} {'Trace Stat':>12} {'Crit (5%)':>12} {'Reject H0?':>12}")
print("-" * 46)
for i in range(len(johansen_result.trace_stat)):
    reject = "Yes" if johansen_result.trace_stat[i] > johansen_result.trace_stat_crit_vals[i, 1] else "No"
    print(f"r ≤ {i:<2} {johansen_result.trace_stat[i]:>12.4f} "
          f"{johansen_result.trace_stat_crit_vals[i, 1]:>12.4f} {reject:>12}")

# --- Deep dive into the cointegrated pair ---

# Find the best cointegrated pair
best_pair = min(eg_results, key=lambda x: x['pval'])
col_a, col_b = best_pair['col_a'], best_pair['col_b']

print(f"\n{'=' * 80}")
print(f"COINTEGRATED PAIR ANALYSIS: {col_a} / {col_b}")
print(f"{'=' * 80}")

# OLS regression for hedge ratio
from numpy.polynomial.polynomial import polyfit
beta_ols = np.polyfit(prices[col_b].values, prices[col_a].values, 1)
hedge_ratio = beta_ols[0]
intercept = beta_ols[1]

spread = prices[col_a] - hedge_ratio * prices[col_b]
spread_mean = spread.mean()
spread_std = spread.std()
z_score = (spread - spread_mean) / spread_std

# Half-life of mean reversion
spread_lag = spread.shift(1).dropna()
spread_diff = spread.diff().dropna()
aligned = pd.concat([spread_diff, spread_lag], axis=1).dropna()
aligned.columns = ['diff', 'lag']
phi = np.polyfit(aligned['lag'], aligned['diff'], 1)[0]
half_life = -np.log(2) / phi if phi < 0 else float('inf')

print(f"  Hedge ratio (β): {hedge_ratio:.4f}")
print(f"  Intercept: {intercept:.4f}")
print(f"  Spread mean: {spread_mean:.4f}")
print(f"  Spread std: {spread_std:.4f}")
print(f"  Mean-reversion φ: {phi:.4f}")
print(f"  Half-life: {half_life:.1f} days")

# ADF on the spread
adf_stat, adf_p, _, _, _, _ = adfuller(spread.dropna())
print(f"  Spread ADF stat: {adf_stat:.4f} (p = {adf_p:.4f})")

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle(f"Cointegration Analysis: {col_a} / {col_b}",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Prices
ax = axes[0]
ax.plot(prices.index, prices[col_a], color=COLORS['asset_a'], label=col_a)
ax2 = ax.twinx()
ax2.plot(prices.index, prices[col_b], color=COLORS['asset_b'], label=col_b)
ax.set_ylabel(f'{col_a} ($)', color=COLORS['asset_a'])
ax2.set_ylabel(f'{col_b} ($)', color=COLORS['asset_b'])
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax.set_title('Price Levels (Non-Stationary)', fontsize=14)

# Panel 2: Spread
ax = axes[1]
ax.plot(spread.index, spread, color=COLORS['spread'], linewidth=1.2)
ax.axhline(spread_mean, color='black', linestyle='-', linewidth=0.8, label='Mean')
ax.axhline(spread_mean + 2 * spread_std, color='red', linestyle='--', linewidth=0.8, label='±2σ')
ax.axhline(spread_mean - 2 * spread_std, color='red', linestyle='--', linewidth=0.8)
ax.fill_between(spread.index,
                spread_mean - 2 * spread_std,
                spread_mean + 2 * spread_std,
                alpha=0.06, color='red')
ax.set_title(f'Cointegrating Spread (Half-Life: {half_life:.0f} days)', fontsize=14)
ax.set_ylabel('Spread')
ax.legend()

# Panel 3: Z-Score
ax = axes[2]
colors_z = np.where(z_score > 0, COLORS['asset_b'], COLORS['asset_a'])
ax.bar(z_score.index, z_score, color=colors_z, alpha=0.5, width=1.0)
ax.axhline(2, color='red', linestyle='--', linewidth=0.8)
ax.axhline(-2, color='red', linestyle='--', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Spread Z-Score (Trading Signal)', fontsize=14)
ax.set_ylabel('Z-Score')
ax.set_xlabel('Date')

plt.tight_layout()
plt.show()

---

## 4. The Kalman Filter: Seeing Through Noise

### 4.1 The Philosophical Motivation

The hedge ratio $\beta$ we estimated above was static—a single number computed over the entire sample. But relationships between assets are not static. They evolve as business conditions change, as new competitors enter markets, as regulatory environments shift. A hedge ratio estimated on last year's data may be stale today. What we need is a way to **continuously update** our estimate of $\beta$ as new data arrives, giving more weight to recent observations while retaining the stabilizing influence of the historical relationship.

The **Kalman filter** is the optimal solution to this problem. Originally developed for aerospace navigation (tracking a spacecraft's position from noisy sensor readings), the Kalman filter maintains a **state estimate** (our hedge ratio) and a **measure of uncertainty** about that estimate. At each time step, it performs two operations:

1. **Predict**: Project the current state forward using a dynamic model
2. **Update**: Incorporate the new observation, weighting the prediction and the observation by their relative uncertainties

### 4.2 The State-Space Representation

For dynamic hedge ratio estimation, we model the observation equation as:

$$P_t^A = \beta_t P_t^B + \alpha_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, R)$$

and the state transition (random walk for parameters):

$$\begin{bmatrix} \alpha_t \\ \beta_t \end{bmatrix} = \begin{bmatrix} \alpha_{t-1} \\ \beta_{t-1} \end{bmatrix} + \boldsymbol{\eta}_t, \quad \boldsymbol{\eta}_t \sim N(\mathbf{0}, Q)$$

The Kalman filter recursively estimates $\alpha_t$ and $\beta_t$, adapting the hedge ratio over time. The matrices $R$ (observation noise) and $Q$ (state noise) control the filter's behavior: large $Q$ relative to $R$ makes the filter responsive (fast-adapting but noisy); small $Q$ makes it smooth (slow-adapting but stable).

### 4.3 Why Kalman Filtering Matters for Trading

The Kalman filter is the bridge between **static models** (which assume relationships are constant) and **reality** (where relationships evolve). In pairs trading, a static hedge ratio that was correct six months ago may cause your spread to drift non-stationarily. The Kalman-estimated hedge ratio tracks the true relationship, keeping your spread stationary and your strategy profitable. Beyond pairs trading, Kalman filters are used for:

- **Dynamic beta estimation** (tracking a stock's market sensitivity over time)
- **Factor loading estimation** (how exposed is a portfolio to each risk factor *right now*?)
- **Signal extraction** (separating a true signal from noisy observations)
- **Regime detection** (detecting shifts in relationships as they happen)

In [ ]:
# ========================================
# Kalman Filter for Dynamic Hedge Ratios
# ========================================

class KalmanFilter:
    """Kalman filter for online linear regression: y_t = alpha_t + beta_t * x_t + eps_t.
    Estimates time-varying intercept and slope (hedge ratio)."""

    def __init__(self, delta: float = 1e-4, ve: float = 1e-3):
        """
        Parameters
        ----------
        delta : float
            State transition covariance scaling. Controls how quickly
            the hedge ratio adapts. Larger = more responsive.
        ve : float
            Observation noise variance estimate.
        """
        self.delta = delta
        self.ve = ve
        self.theta = np.zeros(2)       # [alpha, beta]
        self.P = np.eye(2)             # State covariance
        self.R = np.eye(2) * delta     # State transition noise

        self.history = {'alpha': [], 'beta': [], 'spread': [],
                        'sqrt_Q': [], 'forecast_error': []}

    def update(self, y: float, x: float) -> Dict[str, float]:
        """Process one observation and update state estimates."""
        F = np.array([1.0, x])  # observation vector

        # Predict
        self.P = self.P + self.R

        # Forecast
        y_hat = F @ self.theta
        forecast_error = y - y_hat

        # Innovation covariance
        Q_t = F @ self.P @ F + self.ve

        # Kalman gain
        K = self.P @ F / Q_t

        # Update
        self.theta = self.theta + K * forecast_error
        self.P = self.P - np.outer(K, F) @ self.P

        alpha, beta = self.theta
        spread = y - alpha - beta * x

        self.history['alpha'].append(alpha)
        self.history['beta'].append(beta)
        self.history['spread'].append(spread)
        self.history['sqrt_Q'].append(np.sqrt(Q_t))
        self.history['forecast_error'].append(forecast_error)

        return {'alpha': alpha, 'beta': beta, 'spread': spread,
                'sqrt_Q': np.sqrt(Q_t), 'forecast_error': forecast_error}


# --- Run Kalman filter on the cointegrated pair ---

kf = KalmanFilter(delta=1e-4, ve=1e-3)

for i in range(len(prices)):
    kf.update(y=prices[col_a].iloc[i], x=prices[col_b].iloc[i])

kf_df = pd.DataFrame(kf.history, index=prices.index)

print("=" * 80)
print("KALMAN FILTER RESULTS")
print("=" * 80)
print(f"  Pair: {col_a} / {col_b}")
print(f"  Static OLS hedge ratio: {hedge_ratio:.4f}")
print(f"  Final Kalman hedge ratio: {kf_df['beta'].iloc[-1]:.4f}")
print(f"  Kalman beta range: [{kf_df['beta'].min():.4f}, {kf_df['beta'].max():.4f}]")
print(f"  Kalman spread std: {kf_df['spread'].std():.4f}")

# Test stationarity of Kalman spread vs OLS spread
adf_kalman = adfuller(kf_df['spread'].dropna())
adf_ols = adfuller(spread.dropna())
print(f"\n  Spread stationarity (ADF p-value):")
print(f"    OLS spread:    {adf_ols[1]:.6f}")
print(f"    Kalman spread: {adf_kalman[1]:.6f}")

# --- Visualization ---

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)
fig.suptitle(f"Kalman Filter: Dynamic Hedge Ratio Estimation ({col_a} / {col_b})",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Dynamic hedge ratio
ax = axes[0]
ax.plot(kf_df.index, kf_df['beta'], color=COLORS['kalman'], linewidth=1.5, label='Kalman β(t)')
ax.axhline(hedge_ratio, color=COLORS['neutral'], linestyle='--', linewidth=1.2,
           label=f'Static OLS β = {hedge_ratio:.3f}')
ax.set_title('Time-Varying Hedge Ratio', fontsize=14)
ax.set_ylabel('Hedge Ratio (β)')
ax.legend()

# Panel 2: Kalman spread vs OLS spread
ax = axes[1]
ax.plot(spread.index, spread, color=COLORS['neutral'], linewidth=0.8, alpha=0.5, label='OLS Spread')
ax.plot(kf_df.index, kf_df['spread'], color=COLORS['kalman'], linewidth=1.2, label='Kalman Spread')
ax.axhline(0, color='black', linewidth=0.5)
kalman_std = kf_df['spread'].std()
ax.axhline(2 * kalman_std, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(-2 * kalman_std, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('Spread Comparison: Static OLS vs Dynamic Kalman', fontsize=14)
ax.set_ylabel('Spread')
ax.legend()

# Panel 3: Forecast error (should be white noise)
ax = axes[2]
fe = kf_df['forecast_error']
ax.plot(fe.index, fe, color=COLORS['neutral'], linewidth=0.5, alpha=0.6)
ax.fill_between(kf_df.index, -2 * kf_df['sqrt_Q'], 2 * kf_df['sqrt_Q'],
                alpha=0.15, color=COLORS['confidence'], label='±2σ Forecast Envelope')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Kalman Forecast Errors (Should Be White Noise)', fontsize=14)
ax.set_ylabel('Forecast Error')
ax.set_xlabel('Date')
ax.legend()

plt.tight_layout()
plt.show()

---

## 5. Pairs Trading Strategy: From Theory to P&L

### 5.1 Strategy Design

Pairs trading is the canonical application of cointegration theory. The logic is elegant:

1. **Identify** a cointegrated pair with a mean-reverting spread
2. **Monitor** the spread's z-score (how far it has deviated from equilibrium)
3. **Enter** when the z-score exceeds a threshold: go long the undervalued asset, short the overvalued one
4. **Exit** when the spread reverts to equilibrium (z-score returns near zero)
5. **Stop out** if the spread diverges beyond a wider threshold (the cointegration relationship may have broken)

The Kalman filter improves this strategy by dynamically adjusting the hedge ratio, keeping the spread stationary even as the underlying relationship evolves. We use the Kalman spread's z-score as our trading signal.

### 5.2 Risk Management in Pairs Trading

The primary risk in pairs trading is **cointegration breakdown**—the long-run relationship ceases to exist, and the spread diverges permanently. This can happen when one company undergoes a fundamental change (new CEO, industry disruption, fraud) that severs its economic linkage to its pair partner. Stop-losses are essential: if the spread exceeds, say, ±4 standard deviations, exit the trade and re-evaluate the relationship.

In [ ]:
# ========================================
# Pairs Trading Strategy Backtest
# ========================================

@dataclass
class PairsTradeConfig:
    """Configuration for a pairs trading strategy."""
    entry_z: float = 2.0       # Enter when |z| > entry_z
    exit_z: float = 0.5        # Exit when |z| < exit_z
    stop_z: float = 4.0        # Stop-loss when |z| > stop_z
    lookback: int = 60         # Rolling window for z-score
    use_kalman: bool = True    # Use Kalman filter for hedge ratio


class PairsBacktester:
    """Backtests a pairs trading strategy with proper out-of-sample discipline."""

    def __init__(self, prices_a: pd.Series, prices_b: pd.Series,
                 config: PairsTradeConfig):
        self.prices_a = prices_a
        self.prices_b = prices_b
        self.config = config
        self.results = None

    def run(self) -> pd.DataFrame:
        """Execute the pairs trading backtest."""
        n = len(self.prices_a)
        positions = np.zeros(n)  # +1 = long spread, -1 = short spread, 0 = flat

        if self.config.use_kalman:
            kf = KalmanFilter(delta=1e-4, ve=1e-3)
            spreads = np.zeros(n)
            betas = np.zeros(n)
            for i in range(n):
                result = kf.update(self.prices_a.iloc[i], self.prices_b.iloc[i])
                spreads[i] = result['spread']
                betas[i] = result['beta']
            spread = pd.Series(spreads, index=self.prices_a.index)
            hedge_ratios = pd.Series(betas, index=self.prices_a.index)
        else:
            beta_static = np.polyfit(self.prices_b.values, self.prices_a.values, 1)[0]
            spread = self.prices_a - beta_static * self.prices_b
            hedge_ratios = pd.Series(beta_static, index=self.prices_a.index)

        # Rolling z-score
        rolling_mean = spread.rolling(self.config.lookback, min_periods=20).mean()
        rolling_std = spread.rolling(self.config.lookback, min_periods=20).std()
        z_score = (spread - rolling_mean) / rolling_std

        # Generate signals
        position = 0
        trades = []

        for i in range(self.config.lookback, n):
            z = z_score.iloc[i]

            if np.isnan(z):
                positions[i] = position
                continue

            if position == 0:
                if z > self.config.entry_z:
                    position = -1  # Short spread (spread too high)
                    trades.append(('SHORT', i, z))
                elif z < -self.config.entry_z:
                    position = 1   # Long spread (spread too low)
                    trades.append(('LONG', i, z))
            elif position == 1:
                if z > -self.config.exit_z or z > self.config.stop_z:
                    position = 0
                    trades.append(('EXIT', i, z))
            elif position == -1:
                if z < self.config.exit_z or z < -self.config.stop_z:
                    position = 0
                    trades.append(('EXIT', i, z))

            positions[i] = position

        # Compute returns
        spread_returns = spread.diff() / self.prices_a  # normalized
        strategy_returns = positions * spread_returns.shift(-1)  # next-period return

        self.results = pd.DataFrame({
            'spread': spread,
            'z_score': z_score,
            'position': positions,
            'spread_return': spread_returns,
            'strategy_return': strategy_returns,
            'hedge_ratio': hedge_ratios,
        }, index=self.prices_a.index)

        self.results['cumulative_return'] = (1 + self.results['strategy_return'].fillna(0)).cumprod()
        self.trades = trades

        return self.results

    def compute_metrics(self) -> Dict[str, float]:
        """Compute strategy performance metrics."""
        if self.results is None:
            raise ValueError("Run backtest first.")

        strat_returns = self.results['strategy_return'].dropna()
        strat_returns = strat_returns[strat_returns != 0]

        if len(strat_returns) == 0:
            return {'error': 'No trades executed'}

        cum_ret = self.results['cumulative_return']
        running_max = cum_ret.cummax()
        drawdowns = (cum_ret - running_max) / running_max

        annual_return = strat_returns.mean() * 252
        annual_vol = strat_returns.std() * np.sqrt(252)
        sharpe = annual_return / annual_vol if annual_vol > 0 else 0

        return {
            'Total Return': f"{(cum_ret.iloc[-1] - 1) * 100:.2f}%",
            'Annual Return': f"{annual_return * 100:.2f}%",
            'Annual Volatility': f"{annual_vol * 100:.2f}%",
            'Sharpe Ratio': round(sharpe, 3),
            'Max Drawdown': f"{drawdowns.min() * 100:.2f}%",
            'Num Trades': len([t for t in self.trades if t[0] != 'EXIT']),
            'Win Rate': f"{(strat_returns > 0).mean() * 100:.1f}%",
            'Profit Factor': round(strat_returns[strat_returns > 0].sum() /
                                   abs(strat_returns[strat_returns < 0].sum()), 3)
                             if (strat_returns < 0).any() else float('inf'),
        }


# --- Run backtests: Kalman vs Static ---

config_kalman = PairsTradeConfig(entry_z=2.0, exit_z=0.5, stop_z=4.0,
                                 lookback=60, use_kalman=True)
config_static = PairsTradeConfig(entry_z=2.0, exit_z=0.5, stop_z=4.0,
                                  lookback=60, use_kalman=False)

bt_kalman = PairsBacktester(prices[col_a], prices[col_b], config_kalman)
bt_static = PairsBacktester(prices[col_a], prices[col_b], config_static)

results_kalman = bt_kalman.run()
results_static = bt_static.run()

metrics_kalman = bt_kalman.compute_metrics()
metrics_static = bt_static.compute_metrics()

print("=" * 80)
print("PAIRS TRADING BACKTEST RESULTS")
print("=" * 80)

print(f"\n{'Metric':<22} {'Kalman Filter':>16} {'Static OLS':>16}")
print("-" * 56)
for key in metrics_kalman:
    print(f"{key:<22} {str(metrics_kalman[key]):>16} {str(metrics_static[key]):>16}")

# --- Visualization ---

fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
fig.suptitle(f"Pairs Trading: {col_a} / {col_b}",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Spread with entry/exit zones
ax = axes[0]
ax.plot(results_kalman.index, results_kalman['z_score'],
        color=COLORS['spread'], linewidth=0.9, label='Z-Score')
ax.axhline(config_kalman.entry_z, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-config_kalman.entry_z, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(config_kalman.exit_z, color='green', linestyle=':', linewidth=0.8, alpha=0.6)
ax.axhline(-config_kalman.exit_z, color='green', linestyle=':', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.fill_between(results_kalman.index,
                config_kalman.entry_z, config_kalman.stop_z,
                alpha=0.06, color='red', label='Short Zone')
ax.fill_between(results_kalman.index,
                -config_kalman.entry_z, -config_kalman.stop_z,
                alpha=0.06, color='blue', label='Long Zone')
ax.set_title('Spread Z-Score with Entry/Exit Zones', fontsize=14)
ax.set_ylabel('Z-Score')
ax.legend(loc='upper left', fontsize=9)

# Panel 2: Position
ax = axes[1]
ax.fill_between(results_kalman.index, results_kalman['position'], 0,
                where=results_kalman['position'] > 0,
                color=COLORS['asset_a'], alpha=0.4, label='Long Spread')
ax.fill_between(results_kalman.index, results_kalman['position'], 0,
                where=results_kalman['position'] < 0,
                color=COLORS['asset_b'], alpha=0.4, label='Short Spread')
ax.set_title('Position Over Time', fontsize=14)
ax.set_ylabel('Position')
ax.set_ylim(-1.5, 1.5)
ax.legend(loc='upper left', fontsize=9)

# Panel 3: Cumulative returns comparison
ax = axes[2]
ax.plot(results_kalman.index, (results_kalman['cumulative_return'] - 1) * 100,
        color=COLORS['kalman'], linewidth=2, label='Kalman Filter')
ax.plot(results_static.index, (results_static['cumulative_return'] - 1) * 100,
        color=COLORS['neutral'], linewidth=1.5, linestyle='--', label='Static OLS')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Cumulative Strategy Returns', fontsize=14)
ax.set_ylabel('Return (%)')
ax.legend(loc='upper left')

# Panel 4: Dynamic hedge ratio
ax = axes[3]
ax.plot(results_kalman.index, results_kalman['hedge_ratio'],
        color=COLORS['kalman'], linewidth=1.5, label='Kalman β(t)')
ax.axhline(hedge_ratio, color=COLORS['neutral'], linestyle='--', linewidth=1,
           label=f'Static β = {hedge_ratio:.3f}')
ax.set_title('Hedge Ratio: Kalman vs Static', fontsize=14)
ax.set_ylabel('Hedge Ratio')
ax.set_xlabel('Date')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


# ========================================
# Module Summary
# ========================================

print("\n" + "=" * 80)
print(" " * 18 + "MODULE 3.2 — SUMMARY")
print("=" * 80)
print("""
Key Concepts Covered:

  1. VECTOR AUTOREGRESSION (VAR)
     • Models systems of interacting time series simultaneously
     • Every variable is both predictor and predicted
     • Impulse response functions reveal shock propagation dynamics

  2. GRANGER CAUSALITY
     • Tests whether X helps predict Y beyond Y's own history
     • Predictive content, not true causation
     • Foundation for lead-lag trading strategies

  3. COINTEGRATION
     • Non-stationary series can share a stationary spread
     • Cointegration ≠ correlation: one is about levels, the other about returns
     • Engle-Granger for pairs, Johansen for systems
     • Half-life measures how fast the spread mean-reverts

  4. KALMAN FILTER
     • Optimal online estimation of time-varying parameters
     • Dynamic hedge ratios adapt to evolving relationships
     • Produces more stationary spreads than static estimation
     • Foundation for many real-time trading systems

  5. PAIRS TRADING
     • The canonical cointegration strategy
     • Entry on extreme z-score, exit on mean reversion
     • Kalman-filtered hedge ratios improve robustness
     • Primary risk: cointegration breakdown

  The multivariate perspective transforms trading from a collection
  of independent bets into a coherent view of market structure.
  The relationships between assets are often more predictable—and
  more profitable—than the assets themselves.
""")
print("=" * 80)
print("\n📚 Next: Module 4 — Portfolio Theory & Risk Management\n")